# 03 — Average True Range (ATR)

## Goal
Compute ATR(14) — a **volatility ruler** that makes every threshold adaptive to current market conditions, across all four timeframes.

Instead of hardcoding "2 pips is a big move", we say "1.5× ATR is a big move" — the same algorithm works whether ATR = 0.5 or ATR = 5.0.

---

## Equations

**True Range (TR)** — largest of three distances:

$$\text{TR}_i = \max\!\Bigl(\text{high}_i - \text{low}_i,\;\bigl|\text{high}_i - \text{close}_{i-1}\bigr|,\;\bigl|\text{low}_i - \text{close}_{i-1}\bigr|\Bigr)$$

| Term | Meaning |
|------|---------|
| $\text{high}_i - \text{low}_i$ | Candle's own span |
| $\lvert\text{high}_i - \text{close}_{i-1}\rvert$ | Gap-up: today ran above yesterday's close |
| $\lvert\text{low}_i - \text{close}_{i-1}\rvert$ | Gap-down: today dropped below yesterday's close |

**ATR — Wilder's smoothed moving average** (seed: $\text{ATR}_0 = \text{TR}_0$):

$$\text{ATR}_i = \frac{\text{ATR}_{i-1} \times (n-1) + \text{TR}_i}{n}, \quad n = 14$$

Equivalent to an EMA with $\alpha = 1/n$ — older bars decay exponentially but never vanish.

---

## Why ATR? (The Ruler Analogy)

Every downstream threshold is expressed as a **multiple of ATR**, not a fixed price distance.

| Usage | Condition | Notebook |
|-------|-----------|----------|
| Base tightness | $\dfrac{\text{base\_width}}{\text{avg\_ATR}} \leq 2.5$ | NB04 |
| Leg strength | $\dfrac{\lvert\text{leg\_net}\rvert}{\text{avg\_ATR}} \geq 1.5$ | NB05 |
| Departure strength | $\dfrac{\text{departure}}{\text{avg\_ATR}} \geq 1.5$ | NB06 |


## 1. Setup & load data

In [1]:
import sys
sys.path.append("..")

import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from utils.data_loader import load_all_timeframes
from utils.models import CandlePrimitives, add_atr
from utils.config import (
    ATR_PERIOD,
    COLOR_BULL, COLOR_BEAR,
    CHART_BG as BG, CHART_GRID as GRID,
)

pio.renderers.default = "notebook_connected"

COLOR_ATR = "#7c83fd"
SYMBOL    = "USDJPY=X"   # ← change to any downloaded symbol

# ── load & enrich all 4 TFs ──────────────────────────────────────────────────
raw  = load_all_timeframes(SYMBOL, align=True)
data = {tf: add_atr(CandlePrimitives.enrich_dataframe(df)) for tf, df in raw.items()}
print(f"Loaded {SYMBOL} — {list(data.keys())}")
for tf, df in data.items():
    print(f"  {tf}: {len(df)} rows  |  ATR last: {df['atr'].iloc[-1]:.4f}")


Loaded USDJPY=X — ['1wk', '1d', '4h', '1h']
  1wk: 104 rows  |  ATR last: 2.2481
  1d: 517 rows  |  ATR last: 0.7470
  4h: 3175 rows  |  ATR last: 0.2053
  1h: 12262 rows  |  ATR last: 0.1175


## 2. ATR stats per timeframe

`add_atr()` from `utils/models.py` applies Wilder's recurrence over the `true_range` that `CandlePrimitives.enrich_dataframe()` already computed:

$$\text{ATR}_i = \frac{\text{ATR}_{i-1} \times (n-1) + \text{TR}_i}{n}, \quad n = 14, \quad \text{ATR}_0 = \text{TR}_0$$


In [2]:
# ── quick stats ──────────────────────────────────────────────────────────────
print(f"{'TF':<6} {'Rows':>6}  {'ATR mean':>10}  {'ATR last':>10}")
print("-" * 38)
for tf, df in data.items():
    print(f"{tf:<6} {len(df):>6}  {df['atr'].mean():>10.4f}  {df['atr'].iloc[-1]:>10.4f}")


TF       Rows    ATR mean    ATR last
--------------------------------------
1wk       104      3.1626      2.2481
1d        517      1.5559      0.7470
4h       3175      0.5166      0.2053
1h      12262      0.2572      0.1175


## 3. Inspect — daily sample

In [3]:
data["1d"][["open","high","low","close","true_range","atr"]].tail(10)


,open,high,low,close,true_range,atr
Date,,,,,,
2026-05-21 23:00:00+00:00,159.048004,159.225006,159.000000,159.018005,0.337006,0.999232
2026-05-24 23:00:00+00:00,158.942001,159.037003,158.759003,158.945999,0.278000,0.947716
2026-05-25 23:00:00+00:00,158.947998,159.371994,158.906998,158.953995,0.464996,0.913236
2026-05-26 23:00:00+00:00,159.223999,159.518005,159.173996,159.242996,0.564011,0.888291
2026-05-27 23:00:00+00:00,159.574005,159.643005,159.166000,159.567993,0.477005,0.858913
2026-05-28 23:00:00+00:00,159.261002,159.376007,159.098999,159.270004,0.468994,0.831062
2026-05-31 23:00:00+00:00,159.382996,159.748993,159.367996,159.352997,0.478989,0.805914
2026-06-01 23:00:00+00:00,159.639999,159.895004,159.636002,159.636002,0.542007,0.787063
2026-06-02 23:00:00+00:00,159.975006,159.994003,159.511993,159.968002,0.482010,0.765274


## 4. Visualize ATR on all 4 timeframes

Each chart: **top panel** = candlesticks, **bottom panel** = ATR(14) filled area.

What to observe:
- ATR is **flat and low** during quiet, tight consolidations (future base clusters)
- ATR **climbs** during volatile impulsive moves (future legs)
- ATR **decays slowly** after volatility — Wilder's smoothing keeps memory of past volatility

This single line is what makes every threshold in NB04–NB07 adaptive.


In [4]:
def plot_atr(df, tf: str, n_candles: int = 120):
    d = df.tail(n_candles)

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        row_heights=[0.65, 0.35],
        vertical_spacing=0.03,
        subplot_titles=[f"{SYMBOL} {tf} — Price", f"ATR({ATR_PERIOD})"],
    )

    fig.add_trace(go.Candlestick(
        x=d.index,
        open=d["open"], high=d["high"],
        low=d["low"],   close=d["close"],
        increasing_line_color=COLOR_BULL,
        decreasing_line_color=COLOR_BEAR,
        name="Price",
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=d.index, y=d["atr"],
        mode="lines",
        line=dict(color=COLOR_ATR, width=1.5),
        fill="tozeroy",
        fillcolor="rgba(124,131,253,0.10)",
        name=f"ATR({ATR_PERIOD})",
        hovertemplate="%{x}<br>ATR: %{y:.4f}<extra></extra>",
    ), row=2, col=1)

    fig.update_layout(
        height=520,
        plot_bgcolor=BG, paper_bgcolor=BG, font_color="white",
        xaxis_rangeslider_visible=False,
        hovermode="x unified",
        legend=dict(orientation="h", y=1.06),
    )
    shared = dict(gridcolor=GRID, showgrid=True, zeroline=False)
    for ax in ["xaxis", "xaxis2", "yaxis", "yaxis2"]:
        fig.update_layout(**{ax: shared})
    fig.update_yaxes(title_text="Price",            row=1, col=1)
    fig.update_yaxes(title_text=f"ATR({ATR_PERIOD})", row=2, col=1)
    fig.show()

for tf in ["1wk", "1d", "4h", "1h"]:
    plot_atr(data[tf], tf)


## 5. What's next

| Notebook | Topic |
|----------|-------|
| `04_base_detection.ipynb` | Cluster base candles into zones — uses ATR to guard zone width (`base_width / ATR <= 2.5`) |
| `05_legs_and_formation.ipynb` | Detect the leg-in and leg-out around a base — uses ATR for leg strength (`leg_net / ATR >= 1.5`) |

The `data` dict (all 4 TFs enriched with `true_range` + `atr`) carries forward into NB04.
